# Subsample HRDPS 

This notebook subsample the HRDPS(2.5 km) into HRDPS (10 km).

## Test Subsample

In [2]:
import os
import xarray as xr

HRDPS_path = (
    "/results/forcing/atmospheric/continental2.5/"
    "nemo_forcing/hrdps_y2023m03d01.nc"
)

name_output = "hrdps_subsampled_y2023m03d01.nc"
stride = 4

with xr.open_dataset(HRDPS_path) as ds:
    print("Original dimensions:")
    print(dict(ds.sizes))

    # x、y 方向每 4 个点取 1 个
    ds_subsampled = ds.isel(
        x=slice(None, None, stride),
        y=slice(None, None, stride),
    )

    print("Subsampled dimensions:")
    print(dict(ds_subsampled.sizes))

    # netCDF4 后端支持的 encoding 参数
    valid_encoding_keys = {
        "shuffle",
        "szip_coding",
        "chunksizes",
        "szip_pixels_per_block",
        "blosc_shuffle",
        "quantize_mode",
        "compression",
        "complevel",
        "fletcher32",
        "least_significant_digit",
        "contiguous",
        "significant_digits",
        "endian",
        "_FillValue",
        "dtype",
        "zlib",
    }

    encoding = {}

    for var_name in ds_subsampled.variables:
        original_encoding = ds[var_name].encoding

        # 只保留 netCDF4 后端接受的参数
        var_encoding = {
            key: value
            for key, value in original_encoding.items()
            if key in valid_encoding_keys
        }

        # chunksizes 是针对原始数组形状的，降采样后可能不合适
        var_encoding.pop("chunksizes", None)
        var_encoding.pop("contiguous", None)

        # 使用普通 zlib 压缩，避免继承其他压缩格式
        var_encoding.pop("compression", None)
        var_encoding.pop("szip_coding", None)
        var_encoding.pop("szip_pixels_per_block", None)
        var_encoding.pop("blosc_shuffle", None)

        if ds_subsampled[var_name].ndim > 0:
            var_encoding["zlib"] = True
            var_encoding["complevel"] = 4
            var_encoding["shuffle"] = True

        encoding[var_name] = var_encoding

    ds_subsampled.to_netcdf(
        name_output,
        mode="w",
        format="NETCDF4",
        engine="netcdf4",
        encoding=encoding,
        unlimited_dims=(
            ["time_counter"]
            if "time_counter" in ds_subsampled.dims
            else None
        ),
    )

print(f"Output saved to: {os.path.abspath(name_output)}")

Original dimensions:
{'time_counter': 24, 'y': 230, 'x': 190}
Subsampled dimensions:
{'time_counter': 24, 'y': 58, 'x': 48}
Output saved to: /home/jqiu/analysis-junqi/Analysis_Atmospheric_Forcing/Data_Conversion/Data_HRDPS_Subsample/hrdps_subsampled_y2023m03d01.nc
